# Full Factorial: Illustration of the scaling law of recurrent networks.

The aim of this notebook is to illustrate the effect on training process of hyperparameters changes.

## The Hyperparameters

The hyperparameters that we will consider are:

- Epochs: the number of epochs to train the model.
- Sequence length: axes 1 of the input data, the length of the input sequences. (characters length)
- Number of sequences: axes 0 of the input data, i.e. the number of sequences in the dataset.


### Static Hyperparameters

The following hyperparameters will remain constant throughout the notebook:
- Batch size
- Learning rate (Cosine scheduler applied)
- Token dimension
- Split ratio (train/test)

## Experiment

The analysis are performed through full factorial technique.

## The Training Problem

Given a language grammar, the LSTM will be able to classify sequences of characters as valid or invalid according to the grammar rules.

BNF Definition:

$$
\begin{array}{rcl}
\langle\mathit{string}\rangle   & \mathrel{::=} & \langle\mathit{term}\rangle \\
                              & \mid          & \langle\mathit{string}\rangle \mathbin{\texttt{+}} \langle\mathit{term}\rangle \\[2pt]
\langle\mathit{term}\rangle   & \mathrel{::=} & AB, ED, OK \\[2pt]
\end{array}
$$

In [1]:
"""Static Hyperparameters Configuration"""

BATCH_SIZE = 16
SPLIT_RATIO = 0.9
MAX_LR, MIN_LR = 1e-2, 1e-4

In [2]:
"""Model Architecture"""

from thorcino.activations import Sigmoid
from thorcino.layers.linear import Linear
from thorcino.layers.lstm import LSTM
from thorcino.layers.sequential import Sequential
from thorcino.losses import BinaryCrossEntropyLoss
from thorcino.optimizer import SGD
from thorcino.training.schedulers import CosineSchedule
from thorcino.training.trainer import Trainer

def get_trainer(epochs: int):
    model = Sequential(
        LSTM(
            in_feature=3,
            hidden_units=3,
            out_type='n_to_1',
        ),
        Linear(
            in_feature=3,
            out_feature=1,
        ),
        Sigmoid()
    )
    loss = BinaryCrossEntropyLoss()
    optimizer = SGD(model.parameters, MAX_LR)
    scheduler = CosineSchedule(MAX_LR, MIN_LR, epochs)
    trainer = Trainer(
        model,
        loss,
        optimizer,
        scheduler,
    )

    return trainer

## The Training Process

Different trials will be performed for each hyperparameter, increasing its value by a 40% factor each time, every trails metrics will be plotted and compared to the baseline configuration, showing how the hyperparameter affects the training process and model performance.

In [ ]:

from examples.helpers.dataset import get_dataset, preprocess

def run_experiment(epochs: int, eval_step: int, n_sequence: int, sequence_length: int) -> Trainer:
    X, Y = get_dataset(n_sequence, sequence_length)
    train_dl, test_dl = preprocess(X, Y, BATCH_SIZE, SPLIT_RATIO)

    trainer = get_trainer(epochs)

    for e in range(epochs):
        _ = trainer.train_epoch(train_dl)
        
        if e%eval_step == 0:
            _ = trainer.eval(test_dl)

    return trainer

In [4]:
import matplotlib.pyplot as plt

def plot_metrics(title:str, metrics: dict, epochs: int, eval_step: int):
    fig, (loss_ax, accuracy_ax) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(title)

    loss_ax.plot(range(epochs), metrics['train_loss'], label="train")
    loss_ax.plot(range(0, epochs, eval_step), metrics['eval_loss'], label="test")
    loss_ax.set_title("Losses")
    loss_ax.set_xlabel("epoch")
    loss_ax.set_ylabel("loss")
    loss_ax.legend()

    accuracy_ax.set_title("Accuracy on test sequence")
    accuracy_ax.plot(range(0, epochs, eval_step), metrics['accuracy'], label="accuracy")
    accuracy_ax.set_xlabel("epoch")
    accuracy_ax.set_ylabel("accuracy")
    accuracy_ax.legend()

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
"""Creating Factors"""

import itertools

hyperparams = {
    'number_of_sequence': [100, 200, 400],
    'sequence_length': [10, 20, 40],
    'epochs': [200, 400, 800],
}

## To be a full factorial every factor must has same space size.
first_len = None
for key in hyperparams.keys():
    if first_len == None:
        first_len = len(hyperparams[key])
    assert first_len == len(hyperparams[key]) 

G = itertools.product(range(first_len), repeat=3)

for exp in G:
    idx_n_seq, idx_s_len, idx_epochs = exp
    n_seq, s_len, epochs = hyperparams['number_of_sequence'], hyperparams['sequence_length'], hyperparams['epochs']
    eval_step = int(epochs/10)

    trainer = run_experiment(epochs, eval_step, n_seq, s_len)

0 0 0
0 0 1
0 0 2
0 1 0
0 1 1
0 1 2
0 2 0
0 2 1
0 2 2
1 0 0
1 0 1
1 0 2
1 1 0
1 1 1
1 1 2
1 2 0
1 2 1
1 2 2
2 0 0
2 0 1
2 0 2
2 1 0
2 1 1
2 1 2
2 2 0
2 2 1
2 2 2
